# 6.4.2 Text-to-Image Generation (Diffusion)

Simulate the diffusion forward process on a synthetic 8×8 image. Visualise noise schedules (linear vs cosine) and the denoising trajectory.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)
x0 = rng.standard_normal((8, 8))
T = 1000

def linear_betas(T, b0=1e-4, bT=0.02): return np.linspace(b0, bT, T)
def cosine_betas(T, s=0.008):
    t = np.linspace(0, T, T+1)
    f = np.cos((t/T+s)/(1+s)*np.pi/2)**2
    ab = f/f[0]
    return np.clip(1-ab[1:]/ab[:-1], 1e-5, 0.999)

def forward(x0, t, betas):
    ab = np.cumprod(1-betas)[t]
    return np.sqrt(ab)*x0 + np.sqrt(1-ab)*np.random.default_rng(t).standard_normal(x0.shape)

betas_l = linear_betas(T)
betas_c = cosine_betas(T)
ab_l = np.cumprod(1-betas_l)
ab_c = np.cumprod(1-betas_c)
print(f'Linear  β: {betas_l[0]:.5f} → {betas_l[-1]:.4f}')
print(f'Cosine  β: {betas_c[0]:.5f} → {betas_c[-1]:.4f}')

In [ ]:
timesteps = [0, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(timesteps)+1, figsize=(14, 3))
for j, t in enumerate(timesteps):
    img = forward(x0, t, betas_l)
    axes[j].imshow(img, cmap='gray', vmin=-3, vmax=3)
    axes[j].set(title=f't={t}', xticks=[], yticks=[])

axes[-1].plot(np.arange(T), 1-ab_l, label='Linear', color='steelblue')
axes[-1].plot(np.arange(T), 1-ab_c, label='Cosine', color='tomato')
axes[-1].set(xlabel='t', ylabel='1-ᾱ_t', title='Noise Schedule'); axes[-1].legend()
plt.tight_layout()
plt.savefig('output/diffusion_forward.png')
print('Saved diffusion_forward.png')